In [1]:
from pathlib import Path
import requests
import pandas as pd
import time

# Set paths.
s_path_data = Path(r"C:\2026_06_26_Hackathon\Data")
s_out_csv = s_path_data / "Dallas_City_Owned_Street_Lights.csv"

# Use ArcGIS item ID from the page URL.
item_id = "dc12a9fbeee34e5790932af318254dd4"
item_url = f"https://www.arcgis.com/sharing/rest/content/items/{item_id}"

# Get item metadata.
params = {"f": "json"}
item = requests.get(item_url, params=params).json()

print("Title:", item.get("title"))
print("Type:", item.get("type"))
print("Service URL:", item.get("url"))

service_url = item["url"].rstrip("/")
layer_url = service_url + "/0/query"

# Get total row count.
count_params = {"where": "1=1", "returnCountOnly": "true", "f": "json"}
count_result = requests.get(layer_url, params=count_params).json()
total_rows = count_result["count"]
print("Total rows:", total_rows)

# Download rows in chunks.
all_rows = []
chunk_size = 2000

for offset in range(0, total_rows, chunk_size):
    print("Downloading offset:", offset)
    query_params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "json",
        "resultOffset": offset,
        "resultRecordCount": chunk_size
    }
    data = requests.get(layer_url, params=query_params).json()
    features = data.get("features", [])

    for feature in features:
        row = feature.get("attributes", {}).copy()
        geom = feature.get("geometry", {})
        row["longitude"] = geom.get("x")
        row["latitude"] = geom.get("y")
        all_rows.append(row)

    time.sleep(0.25)

# Save.
df = pd.DataFrame(all_rows)
df.to_csv(s_out_csv, index=False)

print("Rows saved:", len(df))
print("Columns:", len(df.columns))
print("Saved:", s_out_csv)
print(df.head().to_string(index=False))

Title: City Owned Street Lights
Type: Feature Service
Service URL: https://services2.arcgis.com/rwnOSbfKSwyTBcwN/arcgis/rest/services/CityOwnedStreetLights/FeatureServer
Total rows: 10734
Rows saved: 10734
Columns: 66
Saved: C:\2026_06_26_Hackathon\Data\Dallas_City_Owned_Street_Lights.csv
 OBJECTID  BULBS OWNER LETTER  DOWNTOWN Comment  WATT  COG_ID  SignalPole Corner  Label_ID  LED  BlockNum     Street  MaintbyTRN     Symbol FixType PoleType Dept LightType  Loc_ID Loc_Label Mapsco Old_Label  ActivationDate PoleManufacturer  PoleManufYear District  Luminaire  Date_Conv  Metered Schedule  AccountNumber MeterAddress  PoleAge  LightAge MaintResp  TRNTariffBill  TRNMeterBill  FreewayLights ZoneName Comment2  GLNX  GLNY  TXDOTReimb  WattageBill  CouncilDistrict  GLN PrevBulbType  PrevWattage AddToTariffBill MapField LEDConversion  PoleLength  EIARiskScore              ProjectforKPI  OnHIN REP_Report  OriginalSchD ProjectFunding    created_user  created_date last_edited_user  last_edited_dat